# 🎯 Module 14: ML Pipelines

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 14 — ML Pipelines  
> **Start Date:** March 1, 2026  
> **Previous:** 13 — NLP Basics  
> **Next:** 15 — MLflow & Experiment Tracking

---

An **ML Pipeline** chains preprocessing and modeling steps into **one reusable workflow**.

Instead of manually managing preprocessing, scaling, encoding, and model training separately, we wrap them in a **single object** that guarantees:

- Consistency
- Reproducibility
- No data leakage
- Easy cross-validation
- Easy tuning
- Easy deployment

In this notebook we'll cover:

- **Transformers** vs **Estimators**
- `fit()` / `transform()` / `fit_transform()`
- Basic `Pipeline`
- **Numerical** and **categorical** preprocessing
- **`ColumnTransformer`** for mixed data
- Classification & regression pipelines
- Pipelines + **cross-validation**
- Pipelines + **GridSearchCV**
- Pipeline **parameter naming** (`model__C`)
- **Feature selection** and **PCA** inside pipelines
- **Saving / loading** pipelines with `joblib`
- Common mistakes & best practices

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what an ML pipeline is and why it matters
- Distinguish **transformers** from **estimators**
- Use `fit`, `transform`, and `fit_transform` correctly
- Build a basic `Pipeline`
- Handle **missing values**, **numeric**, and **categorical** features
- Build a `ColumnTransformer` for mixed data
- Use pipelines with **cross-validation**
- Tune pipelines with `GridSearchCV` and `RandomizedSearchCV`
- Understand **nested parameter names** like `model__C`
- Add **feature selection** and **PCA** to pipelines
- **Save** and **load** pipelines with `joblib`
- Avoid common pipeline mistakes

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
)

import joblib

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What Is an ML Pipeline?

A **pipeline** is a **sequence of steps** that process data and produce predictions.

```text
Raw Data
   ↓
Imputation
   ↓
Encoding
   ↓
Scaling
   ↓
Model
   ↓
Prediction
```

### In scikit-learn

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression()),
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
```

One object handles **everything**.

---
## 🎯 2. Why ML Pipelines Matter

A simple beginner workflow:

```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
```

But real projects have **many** steps:

```text
Missing Values
      ↓
Encoding
      ↓
Scaling
      ↓
Feature Selection
      ↓
Dimensionality Reduction
      ↓
Model
```

Manually managing all this causes:

- Repetitive code
- **Data leakage**
- Different preprocessing at train vs test time
- Harder cross-validation & tuning
- Harder deployment

Pipelines solve all of this.

---
## 🧠 3. Core Mental Model

Think of a pipeline as an **assembly line**:

```text
Raw Material
     ↓
Machine 1
     ↓
Machine 2
     ↓
Machine 3
     ↓
Final Product
```

In ML:

```text
Raw Data
     ↓
Imputation
     ↓
Encoding
     ↓
Scaling
     ↓
Model
     ↓
Prediction
```

---
## 🆚 4. Manual Workflow vs Pipeline

### Manual (works, but messy)

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
```

### Pipeline (clean)

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression()),
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
```

---
## 🧩 5. Transformers vs Estimators

| Component | Purpose | Methods |
| --- | --- | --- |
| **Transformer** | Changes / transforms data | `fit`, `transform`, `fit_transform` |
| **Estimator** | Learns a prediction task | `fit`, `predict` |
| **Pipeline** | Chains steps together | `fit`, `predict` |

### Common transformers

```text
SimpleImputer
StandardScaler
MinMaxScaler
OneHotEncoder
PolynomialFeatures
PCA
SelectKBest
```

### Common estimators

```text
LogisticRegression
LinearRegression
RandomForestClassifier
RandomForestRegressor
SVC
KNeighborsClassifier
```

---
## 🔧 6. `fit`, `transform`, `fit_transform`

### `fit()`
Learns parameters from data:
```python
scaler.fit(X_train)
# learns mean, std
```

### `transform()`
Applies learned transformation:
```python
scaler.transform(X_test)
# uses the mean/std learned from X_train
```

### `fit_transform()`
Fits and transforms in one step (training only):
```python
X_train_scaled = scaler.fit_transform(X_train)
```

### ⚠️ The critical rule

```text
Training data  → fit_transform()
Test data      → transform() only
```

In [ ]:
# Demonstrate the fit/transform rule
X_tr = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
X_te = np.array([[6.0], [7.0]])

scaler = StandardScaler()

# ✅ CORRECT
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

print('Train scaled mean:', X_tr_scaled.mean().round(6))
print('Train scaled std :', X_tr_scaled.std().round(6))
print('Test scaled (uses train stats):')
print(X_te_scaled.ravel().round(3))
print()
print('Notice: test values are NOT mean 0 — because we did not re-fit on test.')

---
## 🏗️ 7. Basic Pipeline

```python
Pipeline([
    ('step_name', transformer),
    ('step_name', transformer),
    ('model', estimator),
])
```

Each step is a **tuple**: `(name, object)`.

In [ ]:
# Load the Breast Cancer dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)

print('Baseline pipeline accuracy:', round(accuracy_score(y_test, preds), 4))

---
## 🧹 8. Handling Missing Values

```python
SimpleImputer(strategy='median')
```

Common strategies:

- `'mean'` — average
- `'median'` — robust to outliers
- `'most_frequent'` — for categorical
- `'constant'` — fill with a constant value

In [ ]:
# Pipeline with imputation + scaling + model
pipe_impute = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

pipe_impute.fit(X_train, y_train)
preds = pipe_impute.predict(X_test)
print('Pipeline (impute + scale + LR) accuracy:', round(accuracy_score(y_test, preds), 4))

---
## 🔢 9. Numerical vs Categorical Features

Real tabular data usually contains **both**:

```text
Numerical   → age, salary, experience
Categorical → city, department, gender
```

They need **different preprocessing**:

```text
Numerical   → Impute → Scale
Categorical → Impute → OneHotEncode
```

Enter `ColumnTransformer`.

---
## 🏛️ 10. ColumnTransformer

Applies **different transformations to different columns**:

```python
ColumnTransformer([
    ('num', numeric_pipeline,   numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])
```

### The diagram

```text
                 Data
                  │
          ┌───────┴───────┐
          ↓               ↓
     Numerical        Categorical
          ↓               ↓
      Imputation       Imputation
          ↓               ↓
       Scaling         Encoding
          └───────┬───────┘
                  ↓
                Model
```

In [ ]:
# Build a mixed-type tabular dataset
np.random.seed(42)
n = 400

df = pd.DataFrame({
    'age':        np.random.randint(18, 65, n).astype(float),
    'salary':     np.random.randint(20000, 150000, n).astype(float),
    'experience': np.random.randint(0, 25, n).astype(float),
    'city':       np.random.choice(['Islamabad', 'Lahore', 'Karachi'], n),
    'department': np.random.choice(['Engineering', 'Sales', 'HR'], n),
    'purchased':  np.random.randint(0, 2, n),
})

# Inject a few missing values to show imputation
df.loc[df.sample(frac=0.05, random_state=1).index, 'salary'] = np.nan
df.loc[df.sample(frac=0.03, random_state=2).index, 'city'] = np.nan

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print()
print(df.head())

In [ ]:
# Split features and target
numeric_features = ['age', 'salary', 'experience']
categorical_features = ['city', 'department']

X = df[numeric_features + categorical_features]
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
# Numerical pipeline
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

# Combine
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

print('Preprocessor ready.')

In [ ]:
# Full pipeline: preprocessor + model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

full_pipeline.fit(X_train, y_train)
preds = full_pipeline.predict(X_test)

print('Mixed-type pipeline accuracy:', round(accuracy_score(y_test, preds), 4))
print()
print(classification_report(y_test, preds))

---
## 🚫 11. Why We Do NOT Fit on Test Data

### ❌ Wrong
```python
scaler.fit_transform(X_test)   # leaks test info
```

### ✅ Correct
```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
```

The model should **never learn** from the test set — including mean, std, categories, and imputation values.

---
## 💧 12. Data Leakage

### ❌ Wrong

```text
Fit preprocessing on ENTIRE dataset
        ↓
Train/Test Split
```

### ✅ Correct

```text
Train/Test Split
        ↓
Fit preprocessing on TRAIN only
        ↓
Transform TEST with train's learned parameters
```

Pipelines enforce this **automatically**.

In [ ]:
# Demonstrate the leakage impact numerically
from sklearn.datasets import make_classification

X_demo, y_demo = make_classification(n_samples=500, n_features=10, random_state=42)

# ❌ Leaky
X_tr, X_te, y_tr, y_te = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)
leak_scaler = StandardScaler().fit(X_demo)
lr_leak = LogisticRegression(max_iter=1000).fit(leak_scaler.transform(X_tr), y_tr)
acc_leak = lr_leak.score(leak_scaler.transform(X_te), y_te)

# ✅ Correct
ok_scaler = StandardScaler().fit(X_tr)
lr_ok = LogisticRegression(max_iter=1000).fit(ok_scaler.transform(X_tr), y_tr)
acc_ok = lr_ok.score(ok_scaler.transform(X_te), y_te)

print(f'Leaky accuracy  : {acc_leak:.4f}')
print(f'Correct accuracy: {acc_ok:.4f}')
print('\nDifference can be small on clean data; large on messy, real-world data.')

---
## 🔄 13. Pipelines + Cross-Validation

Preprocessing must be fitted **within each fold**.

Pipelines handle this automatically.

```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
```

In [ ]:
# Cross-validation with the full pipeline
scores = cross_val_score(
    full_pipeline, X, y,
    cv=5, scoring='accuracy', n_jobs=-1,
)

print('5-fold CV scores:', scores.round(4))
print(f'Mean            : {scores.mean():.4f}')
print(f'Std             : {scores.std():.4f}')

---
## 🎛️ 14. Pipelines + GridSearchCV

Tune hyperparameters **inside the pipeline** using nested names:

```python
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
}
```

### Nested parameter naming

```text
step_name__parameter_name
```

Example: `model__C`

- `model` → pipeline step name
- `C` → hyperparameter inside that step

In [ ]:
# GridSearchCV over the pipeline
param_grid = {
    'model__C':      [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs'],
}

grid = GridSearchCV(
    full_pipeline, param_grid,
    cv=5, scoring='accuracy', n_jobs=-1,
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV acc: {grid.best_score_:.4f}')
print(f'Test acc   : {accuracy_score(y_test, grid.predict(X_test)):.4f}')

---
## 🔎 15. Nested Pipeline Parameters

For deeper pipelines, parameter names go deeper:

```text
preprocessor → num → scaler → with_mean
```

Parameter name:

```python
'preprocessor__num__scaler__with_mean'
```

Same idea, just more `__` separators.

In [ ]:
# GridSearchCV with nested parameters (deeper names)
nested_grid = {
    'preprocessor__num__scaler': [StandardScaler(), MinMaxScaler()],
    'model__C':                  [0.1, 1, 10],
}

nested = GridSearchCV(
    full_pipeline, nested_grid,
    cv=3, scoring='accuracy', n_jobs=-1,
)
nested.fit(X_train, y_train)

print('Best nested params:', nested.best_params_)
print(f'Best CV acc       : {nested.best_score_:.4f}')

---
## 🎲 16. RandomizedSearchCV

Same pattern as GridSearch, but samples the space:

```python
RandomizedSearchCV(pipeline, param_distributions, n_iter=10, cv=5)
```

Useful when the parameter space is large.

In [ ]:
from scipy.stats import loguniform

param_dist = {
    'model__C': loguniform(1e-3, 1e2),
}

rnd = RandomizedSearchCV(
    full_pipeline, param_dist,
    n_iter=15, cv=5, scoring='accuracy',
    random_state=42, n_jobs=-1,
)
rnd.fit(X_train, y_train)

print('Best params:', rnd.best_params_)
print(f'Best CV acc: {rnd.best_score_:.4f}')

---
## 🎯 17. Feature Selection in Pipelines

```python
Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(k=20)),
    ('model', LogisticRegression()),
])
```

Flow:

```text
Raw Data
   ↓
Preprocessing
   ↓
Feature Selection
   ↓
Model
```

In [ ]:
# Pipeline with feature selection
pipe_fs = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(f_classif, k=5)),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_fs.fit(X_train, y_train)
print('With SelectKBest accuracy:', round(accuracy_score(y_test, pipe_fs.predict(X_test)), 4))

---
## 📐 18. PCA in Pipelines

```python
Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('model', LogisticRegression()),
])
```

Keeps dimensionality reduction **inside** the pipeline — no separate step.

In [ ]:
# PCA inside a pipeline (numeric-only for simplicity)
X_num = df[numeric_features].fillna(df[numeric_features].median())
y_num = df['purchased']

Xtr, Xte, ytr, yte = train_test_split(X_num, y_num, test_size=0.2, random_state=42, stratify=y_num)

pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
pipe_pca.fit(Xtr, ytr)
print('With PCA accuracy:', round(accuracy_score(yte, pipe_pca.predict(Xte)), 4))

---
## 💾 19. Saving & Loading Pipelines

```python
import joblib
joblib.dump(pipeline, 'model_pipeline.pkl')

# later
loaded = joblib.load('model_pipeline.pkl')
loaded.predict(new_data)
```

### ⚠️ Save the WHOLE pipeline, not just the model

- Preprocessing is baked in
- No manual steps during deployment
- Training and inference stay consistent

In [ ]:
# Save and reload the full pipeline
out_path = '/tmp/customer_pipeline.pkl'
joblib.dump(full_pipeline, out_path)

loaded = joblib.load(out_path)

# Simulate a production payload
new_data = pd.DataFrame({
    'age':        [30],
    'salary':     [80000],
    'experience': [4],
    'city':       ['Islamabad'],
    'department': ['Engineering'],
})

print('Prediction:', loaded.predict(new_data)[0])

---
## 📈 20. Regression Pipeline

The same structure works for regression — just swap the estimator.

```python
Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
```

In [ ]:
# Regression pipeline demo on a synthetic dataset
from sklearn.datasets import make_regression

Xr, yr = make_regression(n_samples=400, n_features=5, noise=20, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.2, random_state=42)

pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
pipe_reg.fit(Xtr, ytr)
pred = pipe_reg.predict(Xte)

print(f'MAE : {mean_absolute_error(yte, pred):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(yte, pred)):.3f}')
print(f'R²  : {r2_score(yte, pred):.4f}')

---
## 🆚 21. Model Comparison With Pipelines

Keep preprocessing identical, swap the final estimator:

```text
Same Preprocessing
     ↓
LR / SVM / KNN / RF
```

In [ ]:
# Compare models under identical preprocessing
models = {
    'Logistic': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':      KNeighborsClassifier(n_neighbors=5),
    'SVM':      SVC(kernel='rbf', random_state=42),
    'RF':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

print(f'{"Model":<12} {"Acc":>8} {"CV Mean":>10}')
print('-' * 34)
for name, clf in models.items():
    p = Pipeline([
        ('preprocessor', preprocessor),
        ('model', clf),
    ])
    p.fit(X_train, y_train)
    acc = accuracy_score(y_test, p.predict(X_test))
    cv  = cross_val_score(p, X, y, cv=3, scoring='accuracy', n_jobs=-1).mean()
    print(f'{name:<12} {acc:>8.4f} {cv:>10.4f}')

---
## ⚠️ 22. Common Pipeline Mistakes

### ❌ 1. Fitting on test data
```python
scaler.fit_transform(X_test)
```

### ❌ 2. Preprocessing before splitting
```python
X_processed = preprocessing.fit_transform(X)
X_train, X_test = train_test_split(X_processed, y)
```

### ❌ 3. Forgetting `handle_unknown='ignore'`
New categories at inference → crash.

### ❌ 4. Duplicate preprocessing
```python
X_scaled = scaler.fit_transform(X)
pipeline.fit(X_scaled, y)   # scaler inside pipeline also scales!
```

### ❌ 5. Wrong GridSearch parameter name
```python
'C'          # ❌ wrong
'model__C'   # ✅ right
```

---
## ✅ 23. Pipeline Best Practices

1. **Keep preprocessing inside the pipeline**
2. **Split before fitting**
3. **Use `ColumnTransformer` for mixed data**
4. **Handle unknown categories** (`handle_unknown='ignore'`)
5. **Use pipelines with cross-validation**
6. **Use pipelines with `GridSearchCV`**
7. **Save the whole pipeline** (`joblib.dump`)
8. **Keep training and inference preprocessing identical**

---
## 🏋️ 24. Hands-On Exercises

### Exercise 1 — Basic pipeline
Build a `Pipeline` with `StandardScaler` + `LogisticRegression` on the Breast Cancer dataset. Report accuracy.

### Exercise 2 — Add imputation
Introduce some NaN values and add `SimpleImputer(strategy='median')` before the scaler. Does accuracy change?

### Exercise 3 — Mixed data
Build a `ColumnTransformer` with numeric and categorical features. Train a classifier and report accuracy.

### Exercise 4 — Cross-validation
Use `cross_val_score` on the pipeline. Compare mean CV accuracy against a single train/test split.

### Exercise 5 — GridSearch
Tune `model__C` and `model__solver` with `GridSearchCV`. Report the best params.

### Exercise 6 — Nested parameters
Tune the scaler choice (`StandardScaler` vs `MinMaxScaler`) using `preprocessor__num__scaler`.

### Exercise 7 — Save and load
Save the pipeline with `joblib`, load it, and predict on new data.

### Exercise 8 — Feature selection
Add `SelectKBest` to a pipeline. Which k gives the best accuracy?

---
## 🎤 25. Interview Cheat Sheet

| Question | Answer |
| --- | --- |
| What is an ML pipeline? | A sequence of preprocessing and modeling steps connected into one workflow. |
| Why use pipelines? | Reusability, consistency, leakage control, easy tuning, deployment. |
| Transformer vs estimator? | Transformer changes data; estimator predicts. |
| What does `fit()` do? | Learns parameters from data. |
| What does `transform()` do? | Applies learned transformations. |
| What does `fit_transform()` do? | Fits and transforms in one step. |
| What is `ColumnTransformer`? | Applies different preprocessing to different columns. |
| Why `model__C`? | Access hyperparameter `C` of step named `model`. |
| Can pipelines be tuned? | Yes, via `GridSearchCV` / `RandomizedSearchCV`. |
| Can pipelines be saved? | Yes, with `joblib`. |
| Why save the whole pipeline? | Preserves preprocessing + model together. |
| Main risk avoided by pipelines? | **Data leakage** during CV / tuning. |

---
## 🏁 26. Key Takeaways

### You now understand

- What an ML pipeline is
- Why pipelines matter
- **Transformers** vs **estimators**
- `fit` / `transform` / `fit_transform`
- Basic `Pipeline`
- **Imputation**, **scaling**, **encoding**
- `ColumnTransformer`
- **Classification** & **regression** pipelines
- Pipelines + **cross-validation**
- Pipelines + **GridSearchCV**
- **Nested parameter** names
- **Feature selection** & **PCA** in pipelines
- **Saving & loading** pipelines
- Common mistakes
- Best practices

### The core pattern

```text
DATA → PREPROCESSING → MODEL → PREDICTION
```

In scikit-learn:

```python
Pipeline([
    ('preprocessing', preprocessor),
    ('model', model),
])
```

### 🧭 Golden Rule

> **Build preprocessing and the model as one reproducible workflow instead of treating every step as separate code.**

---

### 🔗 What's Next?

**Module 15 — MLflow & Experiment Tracking**

We'll learn how to **track experiments**, compare runs, register models, and keep our ML work **reproducible and organized** — building directly on top of the pipelines we just learned.